#Rhitmic Trading Journal and Account Tracker


In [1]:
import csv
import datetime
import os
import random  # Just for generating sample data

def create_trading_account_tracker(num_accounts=20, days_back=30):
    """
    Creates a CSV file for tracking multiple trading accounts
    
    Parameters:
    - num_accounts: Number of trading accounts to track
    - days_back: Number of past days to generate sample data for
    """
    # Create directory to store all files
    if not os.path.exists('trading_tracker'):
        os.makedirs('trading_tracker')
    
    # Get today's date and generate a list of dates going back
    today = datetime.date.today()
    dates = [(today - datetime.timedelta(days=i)).strftime('%Y-%m-%d') for i in range(days_back)]
    dates.reverse()  # Start with oldest date
    
    # Generate dashboard file
    dashboard_rows = []
    # Header for dashboard
    dashboard_header = ['Account Name', 'Current Balance', 'Goal Balance', 'Diff from Goal', 
                        'Days Since Payout', 'Max Profit Per Day', 'Highest Profit Day', 
                        'Highest Profit Amount', 'Last Updated']
    dashboard_rows.append(dashboard_header)
    
    # Sample goal balances for each account
    goal_balances = [10000 + (i * 1000) for i in range(num_accounts)]
    
    # Generate files for each account
    for account_idx in range(1, num_accounts + 1):
        account_name = f"Account {account_idx}"
        account_filename = f"trading_tracker/{account_name.replace(' ', '_')}.csv"
        
        # Create individual account CSV
        with open(account_filename, 'w', newline='') as csvfile:
            writer = csv.writer(csvfile)
            
            # Write header
            header = ['Date', 'Opening Balance', 'Deposits', 'Withdrawals', 'Daily P/L', 
                      '% Return', 'Cumulative P/L', 'Profit', 'Days Since Payout', 
                      'Goal Balance', 'Diff from Goal', 'Current Balance', 'Notes']
            writer.writerow(header)
            
            # Sample data for the account
            goal_balance = goal_balances[account_idx-1]
            starting_balance = goal_balance * 0.8  # Start at 80% of goal
            current_balance = starting_balance
            cumulative_pl = 0
            last_payout_date = dates[0]  # Initialize to first date
            highest_profit = 0
            highest_profit_day = dates[0]
            
            # Generate data for each day
            for date in dates:
                # Generate some random values for sample data
                deposits = round(random.uniform(0, 500) if random.random() < 0.2 else 0, 2)
                withdrawals = round(random.uniform(0, 300) if random.random() < 0.1 else 0, 2)
                daily_pl = round(random.uniform(-200, 400), 2)
                
                # Calculate values
                percent_return = round((daily_pl / current_balance) * 100, 2) if current_balance > 0 else 0
                cumulative_pl += daily_pl
                profit = max(0, daily_pl)
                
                # Calculate days since payout
                date_obj = datetime.datetime.strptime(date, '%Y-%m-%d').date()
                last_payout_date_obj = datetime.datetime.strptime(last_payout_date, '%Y-%m-%d').date()
                days_since_payout = (date_obj - last_payout_date_obj).days
                
                # Update balance
                current_balance = round(current_balance + deposits - withdrawals + daily_pl, 2)
                diff_from_goal = round(goal_balance - current_balance, 2)
                
                # Generate a sample note
                notes_options = [
                    "Followed strategy",
                    "Deviated from plan",
                    "Market volatility",
                    "News impact",
                    "Technical issue",
                    ""
                ]
                note = random.choice(notes_options)
                
                # Write the row
                row = [date, round(current_balance - daily_pl - deposits + withdrawals, 2), 
                       deposits, withdrawals, daily_pl, percent_return, 
                       round(cumulative_pl, 2), profit, days_since_payout, 
                       goal_balance, diff_from_goal, current_balance, note]
                writer.writerow(row)
                
                # Check if this is the highest profit day
                if profit > highest_profit:
                    highest_profit = profit
                    highest_profit_day = date
                
                # Occasionally simulate a payout which resets the days counter
                if random.random() < 0.05:
                    last_payout_date = date
        
        # Add summary to dashboard
        dashboard_row = [
            account_name,
            current_balance,
            goal_balance,
            diff_from_goal,
            days_since_payout,
            round(goal_balance * 0.01, 2),  # 1% of goal balance as max daily profit
            highest_profit_day,
            highest_profit,
            dates[-1]  # Last date
        ]
        dashboard_rows.append(dashboard_row)
    
    # Write dashboard file
    with open('trading_tracker/Dashboard.csv', 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        for row in dashboard_rows:
            writer.writerow(row)
    
    # Create a master file with instructions for how to use in Google Sheets
    with open('trading_tracker/README.txt', 'w') as f:
        f.write("TRADING ACCOUNT TRACKER\n")
        f.write("======================\n\n")
        f.write("Files included:\n")
        f.write("1. Dashboard.csv - Overview of all accounts\n")
        for account_idx in range(1, num_accounts + 1):
            f.write(f"{account_idx+1}. Account_{account_idx}.csv - Detailed daily tracking for Account {account_idx}\n")
        
        f.write("\nGOOGLE SHEETS IMPORT INSTRUCTIONS:\n")
        f.write("--------------------------------\n")
        f.write("1. Create a new Google Sheet\n")
        f.write("2. Create a separate sheet for each account plus the dashboard\n")
        f.write("3. For each sheet, go to File > Import > Upload and select the corresponding CSV\n")
        f.write("4. Choose 'Replace data at selected cell' and make sure cell A1 is selected\n")
        f.write("5. Set up formulas in the Dashboard to pull data from individual account sheets\n")
        
        f.write("\nSUGGESTED FORMULAS FOR DASHBOARD:\n")
        f.write("--------------------------------\n")
        f.write("To pull latest balance: =QUERY('Account X'!A:L,\"SELECT L WHERE A = date '\"&TEXT(TODAY(),\"yyyy-mm-dd\")&\"'\",0)\n")
        f.write("To calculate days since payout: =DATEDIF([Last Payout Date Cell], TODAY(), \"D\")\n")
        f.write("To find highest profit day: =QUERY('Account X'!A:H,\"SELECT A WHERE H = \"&MAX('Account X'!H:H)&\"\",0)\n")

    print(f"Trading account tracker files created in the 'trading_tracker' directory.")
    print(f"1 Dashboard file and {num_accounts} individual account files have been generated.")
    print("Import these CSV files into Google Sheets following the instructions in README.txt")

if __name__ == "__main__":
    create_trading_account_tracker()

Trading account tracker files created in the 'trading_tracker' directory.
1 Dashboard file and 20 individual account files have been generated.
Import these CSV files into Google Sheets following the instructions in README.txt
